# Cathey — Evaluation Notebook

Measures the model's **intent-classification accuracy** on text inputs.

Each test case feeds a text string through the full pipeline (`agent.handle()`) and
checks whether the output JSON `type` matches the expected label.

**Four intent types**

| Type | When |
|---|---|
| `direct_command` | User names a device + action ('turn on the light') |
| `needs_clarification` | User describes a feeling without specifying a device ('it's cold') |
| `general_qa` | Non-device question, greeting, or conversational filler |
| `invalid` | No 'Cathey' wake word detected (pre-filter catch) |

**Sections**
1. Bootstrap
2. Starter Test Suite (12 cases across all 4 types — add more in section 3)
3. Evaluation Runner
4. Add Your Own Test Cases


## 1. Bootstrap

Loads the LLM, embedding model, memory, and agent.  
*(Skip if already initialized from `dev_debug.ipynb` in the same kernel.)*

In [1]:
# Bootstrap — run once per kernel session
import sys, os, logging, time
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('faster_whisper').setLevel(logging.ERROR)

# Add project root to path (needed when running from a subdirectory)
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sentence_transformers import SentenceTransformer
from llm_parser import LLMParser
from memory import MemoryManager
from agent import CatheyAgent
from config import EMBED_MODEL_NAME

print('Loading LLM ...')
llm = LLMParser()

print('Loading embedding model ...')
embed = SentenceTransformer(EMBED_MODEL_NAME)
embed.encode('warmup', convert_to_numpy=True)

memory = MemoryManager(
    embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist()
)

# Stub GPIO for non-Pi environments
try:
    from gpio_executor import GPIOExecutor
    gpio = GPIOExecutor()
except Exception:
    gpio = None

# Stub TTS — prints instead of speaking (keeps evaluation output clean)
def _silent_speak(text):
    pass

agent = CatheyAgent(llm=llm, memory=memory, speak=_silent_speak, gpio=gpio)
print('\nAll components ready. Starting evaluation ...')

Loading LLM ...
Loading LLM (TinyLlama/TinyLlama-1.1B-Chat-v1.0) on mps [torch.float16] ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM ready.
Loading embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



All components ready. Starting evaluation ...


## 2. Starter Test Suite

12 pre-defined cases covering all 4 intent types.  
Add your own cases in **Section 4** below — format: `(input_text, expected_type)`.

In [2]:
# ── Starter test suite ───────────────────────────────────────────────────────
# Format: (input_text, expected_type)
# expected_type must be one of:
#   'direct_command'      — wake word detected + device + action
#   'needs_clarification' — wake word detected + vague feeling/comfort
#   'general_qa'          — wake word detected + non-device question / greeting
#   'invalid'             — NO wake word detected (Cathey / fuzzy variant absent)
#
# The runner below mirrors production flow:
#   Step 1: contains_assistant_name() prefilter  ← catches 'invalid'
#   Step 2: llm.parse_unified()                  ← classifies the other 3 types

STARTER_CASES = [
    # ── direct_command — Cathey detected + device + explicit action ──────────
    ("Cathey, turn on the light.",            "direct_command"),
    ("Cathey, turn off the AC.",              "direct_command"),
    ("Cathey, set the AC to 22 degrees.",     "direct_command"),
    ("Cathey, open the curtain halfway.",     "direct_command"),

    # ── needs_clarification — Cathey detected + vague feeling ────────────────
    ("Cathey, I feel cold.",                  "needs_clarification"),
    ("Cathey, it's too bright.",              "needs_clarification"),
    ("Cathey, it's a bit dark.",              "needs_clarification"),

    # ── general_qa — Cathey detected + non-device question or greeting ───────
    ("Cathey, hello!",                        "general_qa"),
    ("Cathey, how do I make tea?",            "general_qa"),
    ("Cathey, my name is Alex.",              "general_qa"),

    # ── invalid — NO 'Cathey' wake word (or any fuzzy variant) ───────────────
    # These are caught by the prefilter before even reaching the LLM.
    ("Turn on the light.",                    "invalid"),
    ("It's too cold in here.",                "invalid"),
    ("I feel cold.",                          "invalid"),
    ("How do I make tea?",                    "invalid"),
]

print(f"Starter cases loaded: {len(STARTER_CASES)}")
from collections import Counter
dist = Counter(label for _, label in STARTER_CASES)
for t, n in sorted(dist.items()):
    print(f"  {t:<25} {n} case{'s' if n > 1 else ''}")

Starter cases loaded: 12
  direct_command            4 cases
  general_qa                3 cases
  invalid                   2 cases
  needs_clarification       3 cases
Prefilter cases loaded: 3


## 3. Evaluation Runner

Runs **all cases** (starter + any you added in section 4) through the pipeline
and prints accuracy per intent type and overall.

In [7]:
# ── Evaluation runner ────────────────────────────────────────────────────────
# Mirrors exact production flow:
#   1. contains_assistant_name() prefilter → 'invalid' if no wake word found
#   2. llm.parse_unified(context='')       → LLM classifies the remaining 3 types
# No episodes are saved during evaluation (context='' avoids memory interference).
import time, pandas as pd
from agent import contains_assistant_name

try:
    all_cases = STARTER_CASES + USER_CASES
    print(f"Running {len(all_cases)} cases "
          f"({len(STARTER_CASES)} starter + {len(USER_CASES)} added)")
except NameError:
    all_cases = STARTER_CASES
    print(f"Running {len(all_cases)} starter cases "
          "(define USER_CASES in section 4 to add more)")

print()
rows = []

for text, expected in all_cases:
    t0 = time.perf_counter()

    # ── Step 1: prefilter (wake-word check) ───────────────────────────────────
    if not contains_assistant_name(text):
        got = 'invalid'
        raw = '(prefilter blocked — no wake word)'
        ms  = (time.perf_counter() - t0) * 1000
    else:
        # ── Step 2: LLM classification (no memory context for clean eval) ─────
        result, raw, _ = llm.parse_unified(text, context='', verbose=False)
        got = result.get('type', 'invalid')
        ms  = (time.perf_counter() - t0) * 1000

    match = got == expected
    rows.append({
        'input':      text,
        'expected':   expected,
        'got':        got,
        'match':      match,
        'latency_ms': round(ms, 1),
    })
    mark = '✓' if match else '✗'
    print(f"{mark} [{expected:<22}] → [{got:<22}]  {text[:60]}")

df = pd.DataFrame(rows)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("RESULTS SUMMARY")
print("═" * 70)

for intent_type in ['direct_command', 'needs_clarification', 'general_qa', 'invalid']:
    subset = df[df['expected'] == intent_type]
    if len(subset) == 0:
        continue
    acc    = subset['match'].mean()
    avg_ms = subset['latency_ms'].mean()
    bar    = '█' * int(acc * 20) + '░' * (20 - int(acc * 20))
    print(f"  {intent_type:<22}  [{bar}]  {acc:5.0%}  "
          f"({subset['match'].sum()}/{len(subset)})  avg {avg_ms:.0f} ms")

overall_acc = df['match'].mean()
overall_ms  = df['latency_ms'].mean()
print("─" * 70)
print(f"  {'OVERALL':<22}  {'':22}  {overall_acc:5.0%}  "
      f"({df['match'].sum()}/{len(df)})  avg {overall_ms:.0f} ms")
print("═" * 70)

failures = df[~df['match']]
if len(failures) > 0:
    print(f"\nFailed cases ({len(failures)}):")
    for _, row in failures.iterrows():
        print(f"  ✗ expected={row['expected']:<22} got={row['got']:<22} "
              f"input={row['input'][:55]}")
else:
    print("\n✓ All cases passed!")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running 20 cases (12 starter + 8 added)



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ [direct_command        ] → [direct_command        ]  Cathey, turn on the light.


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

✓ [general_qa            ] → [general_qa            ]  Cathey, what year is it?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ [general_qa            ] → [general_qa            ]  Cathey, can I eat this after two days?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✗ [invalid               ] → [direct_command        ]  Open the window.
✓ [invalid               ] → [invalid               ]  How's the weather today?

══════════════════════════════════════════════════════════════════════
RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════
  direct_command          [████████████████████]   100%  (6/6)  avg 1709 ms
  needs_clarification     [████████████░░░░░░░░]    60%  (3/5)  avg 1775 ms
  general_qa              [████████████████████]   100%  (5/5)  avg 1215 ms
  invalid                 [██████████░░░░░░░░░░]    50%  (2/4)  avg 2499 ms
──────────────────────────────────────────────────────────────────────
  OVERALL                                           80%  (16/20)  avg 1760 ms
══════════════════════════════════════════════════════════════════════

Failed cases (4):
  ✗ expected=invalid                got=direct_command         input=Cathey, .
  ✗ expected=needs_clarification    got=direct_command         inpu

## 4. Add Your Own Customized Test Cases

Define `USER_CASES` below, then re-run the **Evaluation Runner** cell (section 3).

```python
USER_CASES = [
    ("Cathey, close the curtain.",        "direct_command"),
    ("Cathey, I feel a bit hot.",         "needs_clarification"),
    ("Cathey, what year is it?",          "general_qa"),
    ("Open the window.",                  "invalid"),
    # ... add as many as you like
]
```

In [6]:
# ── Your test cases — edit here and re-run section 3 ─────────────────────────
USER_CASES = [
    # direct_command
    ("Cathey, close the curtain.",            "direct_command"),
    ("Cathey, set brightness to 80.",          "direct_command"),

    # needs_clarification
    ("Cathey, I feel a bit hot.",              "needs_clarification"),
    ("Cathey, it's stuffy in here.",           "needs_clarification"),

    # general_qa
    ("Cathey, what year is it?",               "general_qa"),
    ("Cathey, can I eat this after two days?", "general_qa"),

    # invalid (no wake word)
    ("Open the window.",                       "invalid"),
    ("How's the weather today?",               "invalid"),
]

print(f"{len(USER_CASES)} user-added case(s) defined.")
print("Re-run the Evaluation Runner cell (section 3) to include these.")

8 user-added case(s) defined.
Re-run the Evaluation Runner cell (section 3) to include these.
